In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Ensure clean folders exist for artifacts
os.makedirs("data/processed", exist_ok=True)
os.makedirs("reports/figures", exist_ok=True)

# ----------------------------------------------------
# 1. LOAD AND JOIN IMPACT DATA (Task 3.1)
# ----------------------------------------------------
try:
    df_unified = pd.read_csv("data/processed/ethiopia_fi_unified_data_enriched.csv")
    df_links = pd.read_csv("data/processed/impact_links_enriched.csv")
    print("Enriched datasets successfully loaded.")
except FileNotFoundError:
    print("Processed datasets not found. Constructing compliant fallback simulation data...")
    # Generate mock events
    events_mock = pd.DataFrame([
        {'id': 'EV_TELEBIRR_2021', 'record_type': 'event', 'indicator': 'Telebirr Commercial Launch', 'observation_date': '2021-05-11'},
        {'id': 'EV_SAFARICOM_2022', 'record_type': 'event', 'indicator': 'Safaricom GSM Commercial Entry', 'observation_date': '2022-08-29'},
        {'id': 'EV_MPESA_2023', 'record_type': 'event', 'indicator': 'M-Pesa Mobile Money Launch', 'observation_date': '2023-08-15'},
        {'id': 'EV_FAYDA_SCALE_2025', 'record_type': 'event', 'indicator': 'Fayda National Digital ID Integration', 'observation_date': '2025-09-15'}
    ])
    # Generate mock impact links
    df_links = pd.DataFrame([
        {'parent_id': 'EV_TELEBIRR_2021', 'pillar': 'access', 'related_indicator': 'ACC_MM_ACCOUNT', 'impact_direction': 'positive', 'impact_magnitude': 0.08, 'lag_months': 6, 'evidence_basis': 'High initial marketing spend and utility bill integrations.'},
        {'parent_id': 'EV_TELEBIRR_2021', 'pillar': 'usage', 'related_indicator': 'USG_DIGITAL_PAYMENT', 'impact_direction': 'positive', 'impact_magnitude': 0.15, 'lag_months': 3, 'evidence_basis': 'Direct utility payment mandates.'},
        {'parent_id': 'EV_MPESA_2023', 'pillar': 'access', 'related_indicator': 'ACC_MM_ACCOUNT', 'impact_direction': 'positive', 'impact_magnitude': 0.04, 'lag_months': 9, 'evidence_basis': 'Safaricom distribution network leverage.'},
        {'parent_id': 'EV_FAYDA_SCALE_2025', 'pillar': 'access', 'related_indicator': 'ACC_OWNERSHIP', 'impact_direction': 'positive', 'impact_magnitude': 0.12, 'lag_months': 12, 'evidence_basis': 'Elimination of paper ID KYC hurdles.'}
    ])
    df_unified = events_mock

# Filter events and join
events_only = df_unified[df_unified['record_type'] == 'event'].copy()
joined_impacts = pd.merge(df_links, events_only, left_on='parent_id', right_on='id', how='inner')

print("\n=== Joined Impact Links Summary ===")
print(joined_impacts[['parent_id', 'related_indicator', 'impact_magnitude', 'lag_months']])

# ----------------------------------------------------
# 2. EVENT-INDICATOR ASSOCIATION MATRIX (Task 3.2)
# ----------------------------------------------------
# Pivot impact links to create a clean visual coefficients matrix
association_pivot = df_links.pivot_table(
    index='parent_id', 
    columns='related_indicator', 
    values='impact_magnitude', 
    aggfunc='mean'
).fillna(0.0)

# Ensure columns reflect the target indicators required for forecasting
target_cols = ['ACC_OWNERSHIP', 'ACC_MM_ACCOUNT', 'USG_DIGITAL_PAYMENT']
for col in target_cols:
    if col not in association_pivot.columns:
        association_pivot[col] = 0.0

association_matrix = association_pivot[target_cols]

# Save heatmap visualization
plt.figure(figsize=(8, 4.5))
sns.heatmap(association_matrix, annot=True, cmap="Purples", fmt=".2f", cbar=True, linewidths=1, linecolor='gray')
plt.title("Event-Indicator Impact Association Matrix (Max Magnitude)", fontsize=11, fontweight='bold', pad=15)
plt.ylabel("Event Parent ID")
plt.xlabel("Related Inclusion Indicator")
plt.tight_layout()
plt.savefig("reports/figures/task3_association_heatmap.png")
plt.close()
print("\nHeatmap visualization saved to 'reports/figures/task3_association_heatmap.png'")

# ----------------------------------------------------
# 3. MATHEMATICAL EVENT DECAY FUNCTION (Task 3.2)
# ----------------------------------------------------
def calculate_lagged_impact(t, t_event, max_impact, lag_months, lambda_decay=0.08):
    """
    Computes time-dependent event impact using an exponential growth-absorption curve.
    Formula: I(t) = M * (1 - e^(-lambda * (t - t0 - L))) for t >= t0 + L
    """
    # Convert dates or year points to monthly steps
    dt = (t - t_event) * 12 # Years to months conversion
    lagged_dt = dt - lag_months
    
    # If time is before the transmission lag window, impact is 0
    impact = np.where(lagged_dt >= 0, max_impact * (1 - np.exp(-lambda_decay * lagged_dt)), 0.0)
    return impact

# Test and show timeline of Telebirr impact building over months
timeline_years = np.arange(2021, 2026, 0.1)
telebirr_impact_profile = calculate_lagged_impact(
    t=timeline_years, 
    t_event=2021.36,       # May 2021 Launch
    max_impact=0.15,       # 15 percentage point maximum target impact
    lag_months=3,          # 3-month deployment lag
    lambda_decay=0.05      # Gradual market penetration pace
)

plt.figure(figsize=(10, 4))
plt.plot(timeline_years, telebirr_impact_profile * 100, color='#117a65', linewidth=2.5)
plt.axvline(x=2021.36, color='red', linestyle=':', label="Telebirr Launch (May 2021)")
plt.axvline(x=2021.36 + (3/12), color='orange', linestyle=':', label="Lag Threshold (Aug 2021)")
plt.title("Modeled Temporal Absorption: Telebirr Impact on Digital Payments Usage", fontsize=11, fontweight='bold')
plt.xlabel("Year timeline")
plt.ylabel("Model Shock Contribution (% Points)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("reports/figures/task3_telebirr_absorption_curve.png")
plt.close()
print("Temporal decay impact timeline plot saved successfully.")